# Problem Statement : Spam or ham?

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
spam = pd.read_csv('spam.csv')
spam.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
spam.tail()

,Category,Message
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [ ]:
spam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [ ]:
spam.isna().sum()

Category    0
Message     0
dtype: int64

In [ ]:
spam.shape

(5572, 2)

## Feature Extraction : CountVectorizer

In [ ]:
train_set = {"The sky is blue.", "The sun is bright."}
test_set = {"The sun in the sky is bright.","we can see the shining sun, the bright sun."}

In [ ]:
vectorizer = CountVectorizer()

In [ ]:
vectorizer.fit(train_set)
vectorizer.vocabulary_

{'the': 5, 'sun': 4, 'is': 2, 'bright': 1, 'sky': 3, 'blue': 0}

- keys : unique words in the training sentences
- values : indices based on their appearance order when the vocabulary was built

In [ ]:
# we can sort the doctionary
voacb_dict = vectorizer.vocabulary_.copy()
dict(sorted(voacb_dict.items(), key=lambda x: x[1]))

{'blue': 0, 'bright': 1, 'is': 2, 'sky': 3, 'sun': 4, 'the': 5}

In [ ]:
test_set

{'The sun in the sky is bright.',
 'we can see the shining sun, the bright sun.'}

In [ ]:
test_vec = vectorizer.transform(test_set)
print(test_vec)

  (0, 1)	1
  (0, 4)	2
  (0, 5)	2
  (1, 1)	1
  (1, 2)	1
  (1, 3)	1
  (1, 4)	1
  (1, 5)	2


- Each entry is represented as `(row_index, column_index)` and `count`
- `row_index`: Indicates which document (from test_set) the entry corresponds to.
- `column_index`: Indicates which word (from the training vocabulary) is being counted.
- `count`: Indicates how many times the word appears in that document.

In [ ]:
# convert to array
test_vec.toarray()

array([[0, 1, 0, 0, 2, 2],
       [0, 1, 1, 1, 1, 2]], dtype=int64)

In [ ]:
# inverse transform
vectorizer.inverse_transform(test_vec)

[array(['bright', 'sun', 'the'], dtype='<U6'),
 array(['bright', 'is', 'sky', 'sun', 'the'], dtype='<U6')]

We only got the token back, not the oreder of words. This is limitation of coutvectorizer.

## Data processing

In [ ]:
spam.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
# Convierting category into binary column
spam['Category']=spam['Category'].apply(lambda x:1 if x=='spam' else 0)
spam.head(5)

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
X = spam['Message']
y = spam['Category']

In [ ]:
X.shape, y.shape

((5572,), (5572,))

In [ ]:
X.head()

0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: Message, dtype: object

In [ ]:
y.head()

0    0
1    0
2    1
3    0
4    0
Name: Category, dtype: int64

In [ ]:
## apply count vectorizer
vect = CountVectorizer(ngram_range=(1,3),min_df=5,max_features=8000)
X_vect = vect.fit_transform(X).toarray()
X_vect[:5]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [ ]:
vect.fit_transform??

Signature: vect.fit_transform(raw_documents, y=None)
Source:   
    @_fit_context(prefer_skip_nested_validation=True)
    def fit_transform(self, raw_documents, y=None):
        """Learn the vocabulary dictionary and return document-term matrix.

        This is equivalent to fit followed by transform, but more efficiently
        implemented.

        Parameters
        ----------
        raw_documents : iterable
            An iterable which generates either str, unicode or file objects.

        y : None
            This parameter is ignored.

        Returns
        -------
        X : array of shape (n_samples, n_features)
            Document-term matrix.
        """
        # We intentionally don't call the transform method to make
        # fit_transform overridable without unwanted side effects in
        # TfidfVectorizer.
        if isinstance(raw_documents, str):
            raise ValueError(
                "Iterable over raw text documents expected, string object received

In [ ]:
X_vect.shape

(5572, 4722)

4000+ Features got created by vectorizer.

In [ ]:
# split data
X_train, X_test, y_train, y_test = train_test_split(X_vect, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4457, 4722), (1115, 4722), (4457,), (1115,))

## Build Naive Bayes Model

In [ ]:
nb = GaussianNB()
nb.fit(X_train, y_train)

GaussianNB()

In [ ]:
# training score
nb.score(X_train, y_train)

0.9479470495849226

In [ ]:
# test score
nb.score(X_test, y_test)

0.9452914798206278

## Comparing Naive Bayes with KNN and SVM

In [ ]:
# build KNN

%time

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
knn.score(X_train, y_train), knn.score(X_test, y_test)

CPU times: total: 0 ns
Wall time: 0 ns


(0.9367287413058111, 0.9291479820627803)

In [ ]:
# build SVM

#%time
#svm = SVC()
#svm.fit(X_train, y_train)
#svm.score(X_train, y_train), svm.score(X_test, y_test)

In [ ]:
# build Naive Bayes

%time
nb = GaussianNB()
nb.fit(X_train, y_train)
nb.score(X_train, y_train), nb.score(X_test, y_test)

CPU times: total: 0 ns
Wall time: 0 ns


(0.9479470495849226, 0.9452914798206278)

## Prediction

In [ ]:
sample = "You won 25 lakh"
vec = vect.transform([sample]).toarray()
nb.predict(vec)

array([1], dtype=int64)

Correct Classification

In [ ]:
sample = "Hi Rushikesh, I need your help"
vec1 = vect.transform([sample]).toarray()
nb.predict(vec1)

array([1], dtype=int64)

Misclasification, this shows even if we have training and testing score of 94%, we can get wrong results.

## Evaluation Metrics

In [ ]:
# Multinomial Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train,y_train)

MultinomialNB()

In [ ]:
nb.score(X_train,y_train), nb.score(X_test,y_test)

(0.988332959389724, 0.9883408071748879)

In [ ]:
# test sample email
email_1='Sounds great! Are you home now?'

In [ ]:
# Predicting the class of the example emails
vec = vect.transform([email_1]).toarray()
predictions = nb.predict(vec)
print(predictions)

[0]


In [ ]:
email_2 = "Congratulations! You've been selected to receive a FREE iPhone—click here to claim your prize!"

In [ ]:
# Predicting the class of the example emails
vec = vect.transform([email_2]).toarray()
predictions = nb.predict(vec)
print(predictions)

[1]


In [ ]:
sample_1 = "Hi Rushikesh, I need your help"

In [ ]:
# Predicting the class of the example emails
vec = vect.transform([sample_1]).toarray()
predictions = nb.predict(vec)
print(predictions)

[0]


In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, nb.predict(X_test))

array([[965,   1],
       [ 12, 137]], dtype=int64)

In [ ]:
#accuracy
from sklearn.metrics import accuracy_score
accuracy_score(y_test, nb.predict(X_test))

0.9883408071748879

In [ ]:
#precision
from sklearn.metrics import precision_score
precision_score(y_test, nb.predict(X_test))

0.9927536231884058

In [ ]:
#recall
from sklearn.metrics import recall_score
recall_score(y_test, nb.predict(X_test))

0.9194630872483222

In [ ]:
#f1 score
from sklearn.metrics import f1_score
f1_score(y_test, nb.predict(X_test))

0.9547038327526133

- `Accuracy` : This indicates that **98%** of the total predictions made by the model are correct.
- `Precision` : A precision of 99% means that when the model predicts an email as spam, it is correct **99% of the time**.
- `Recall` : A recall of 91% means that the model successfully identifies 91% of the actual spam emails but **misses 9%**.
- `F1 Score` : A score of 95% indicates that the model is performing well in balancing the trade-off between precision and recall.

In [ ]:
# confusion metrics report
from sklearn.metrics import classification_report
print(classification_report(y_test, nb.predict(X_test)))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

